# **1. DATA SOURCE**

Mengambil data weather dari []

Mengambil data berita cuaca dari []

In [1]:
# ini buat requirement installation
 
# !pip install kafka-python requests feedparser
import json
import time
import hashlib
import threading
import requests
import feedparser
from datetime import datetime
from kafka import KafkaProducer, KafkaAdminClient, KafkaConsumer
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

In [ ]:
#buat ambil data dari datasource (berita)

BOOTSTRAP_SERVERS = ["localhost:9092"]

KOTA = [
    {"kode": "JKT", "nama": "Jakarta",  "lat": -6.21, "lon": 106.85},
    {"kode": "SBY", "nama": "Surabaya", "lat": -7.25, "lon": 112.75},
    {"kode": "SMG", "nama": "Semarang", "lat": -6.99, "lon": 110.42},
    {"kode": "MDN", "nama": "Medan",    "lat": -3.59, "lon": 98.67},
    {"kode": "MKS", "nama": "Makassar", "lat": -5.14, "lon": 119.41},
    {"kode": "DPS", "nama": "Denpasar", "lat": -8.67, "lon": 115.21},
]

RSS_URLS = [
   "https://www.antaranews.com/rss/warta-bumi.xml",
"https://www.mongabay.co.id/feed/",
]

TOPIC_API = "weather-api"
TOPIC_RSS = "weather-rss"

INTERVAL_API = 600   # 10 menit
INTERVAL_RSS = 300   # 5 menit

# Stop flag — panggil stop_flag.set() dari sel manapun untuk hentikan semua thread
stop_flag = threading.Event()

print(" Konfigurasi selesai")
print(f"   Kota dipantau : {[k['kode'] for k in KOTA]}")
print(f"   Topic API     : {TOPIC_API}")
print(f"   Topic RSS     : {TOPIC_RSS}")

 Konfigurasi selesai
   Kota dipantau : ['JKT', 'SBY', 'SMG', 'MDN', 'MKS', 'DPS']
   Topic API     : weather-api
   Topic RSS     : weather-rss


In [3]:
#buat ambil data dari datasource (api cuaca) + tes koneksi API

def ambil_cuaca(kota: dict) -> dict | None:
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": kota["lat"],
        "longitude": kota["lon"],
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
        "timezone": "Asia/Jakarta",
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()["current"]
        return {
            "kode_kota": kota["kode"],
            "nama_kota": kota["nama"],
            "temperature": data["temperature_2m"],
            "humidity":    data["relative_humidity_2m"],
            "wind_speed":  data["wind_speed_10m"],
            "weather_code": data["weather_code"],
            "timestamp":   datetime.now().isoformat(),
        }
    except Exception as e:
        print(f"  [ERROR] {kota['nama']}: {e}")
        return None

# Test: ambil data semua kota sekali
print("Test fetch Open-Meteo:\n")
for kota in KOTA:
    event = ambil_cuaca(kota)
    if event:
        print(f"  {event['kode_kota']} | {event['nama_kota']:10s} | "
              f"{event['temperature']}°C | "
              f" {event['wind_speed']} km/h | "
              f" {event['humidity']}%")

Test fetch Open-Meteo:

  JKT | Jakarta    | 31.0°C |  5.6 km/h |  67%
  SBY | Surabaya   | 28.8°C |  9.1 km/h |  78%
  SMG | Semarang   | 30.4°C |  7.6 km/h |  71%
  MDN | Medan      | 28.8°C |  14.1 km/h |  78%
  MKS | Makassar   | 31.4°C |  7.4 km/h |  59%
  DPS | Denpasar   | 28.4°C |  10.0 km/h |  79%


In [4]:
# buat test koneksi RSS single fetch, tidak looping

feedparser.USER_AGENT = "Mozilla/5.0 (compatible; WeatherPulse/1.0)"

def hash_url(url: str) -> str:
    return hashlib.md5(url.encode()).hexdigest()[:8]

def fetch_rss_sekali(url: str) -> list[dict]:
    hasil = []
    try:
        feed = feedparser.parse(url)
        print(f"  Status  : {feed.get('status', 'N/A')}")
        print(f"  Feed    : {feed.feed.get('title', 'tidak ada judul')}")
        print(f"  Entries : {len(feed.entries)} artikel")
        for entry in feed.entries[:3]:  # tampilkan 3 pertama
            link = entry.get("link", "")
            artikel = {
                "judul":       entry.get("title", ""),
                "link":        link,
                "ringkasan":   entry.get("summary", "")[:200],
                "waktu_terbit": entry.get("published", datetime.now().isoformat()),
                "sumber":      feed.feed.get("title", url),
                "timestamp":   datetime.now().isoformat(),
            }
            hasil.append(artikel)
    except Exception as e:
        print(f"  [ERROR] {e}")
    return hasil

# Test semua RSS URL
for url in RSS_URLS:
    print(f"\nTest RSS: {url}")
    artikel = fetch_rss_sekali(url)
    for a in artikel:
        print(f"  → {a['judul'][:70]}...")


Test RSS: https://rss.tempo.co/tag/cuaca
  Status  : 404
  Feed    : tidak ada judul
  Entries : 0 artikel

Test RSS: https://rss.kompas.com/feed/kompas.com/sains/environment
  Status  : 404
  Feed    : tidak ada judul
  Entries : 0 artikel


# **2. DATA INGEST**

In [5]:
#buat masukin data yang diambil ke kafka dan tes koneksi Kafka

producer_api = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    key_serializer=lambda k: k.encode("utf-8"),
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    enable_idempotence=True,
    acks="all",
)

def loop_producer_api():
    print("[API Producer] Dimulai")
    while not stop_flag.is_set():
        print(f"\n[API Producer] [{datetime.now().strftime('%H:%M:%S')}] Polling...")
        for kota in KOTA:
            if stop_flag.is_set():
                break
            event = ambil_cuaca(kota)
            if event:
                producer_api.send(TOPIC_API, key=kota["kode"], value=event)
                print(f"   {event['kode_kota']} | {event['temperature']}°C | "
                      f" {event['wind_speed']} km/h |  {event['humidity']}%")
        producer_api.flush()
        # tunggu interval, tapi cek stop_flag tiap 5 detik
        for _ in range(INTERVAL_API // 5):
            if stop_flag.is_set():
                break
            time.sleep(5)
    print("[API Producer] Berhenti")

thread_api = threading.Thread(target=loop_producer_api, daemon=True, name="ProducerAPI")
thread_api.start()
print(f" Thread producer API berjalan: {thread_api.is_alive()}")

[API Producer] Dimulai

[API Producer] [10:37:30] Polling...
 Thread producer API berjalan: True


In [6]:
#buat masukin data yang diambil ke kafka dan tes koneksi Kafka

producer_rss = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    key_serializer=lambda k: k.encode("utf-8"),
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    enable_idempotence=True,
    acks="all",
)

sudah_dikirim = set()  # deduplication

def loop_producer_rss():
    print("[RSS Producer] Dimulai")
    while not stop_flag.is_set():
        print(f"\n[RSS Producer] [{datetime.now().strftime('%H:%M:%S')}] Polling...")
        total_baru = 0
        for url in RSS_URLS:
            try:
                feed = feedparser.parse(url)
                for entry in feed.entries:
                    link = entry.get("link", "")
                    if not link or link in sudah_dikirim:
                        continue
                    artikel = {
                        "judul":        entry.get("title", ""),
                        "link":         link,
                        "ringkasan":    entry.get("summary", "")[:300],
                        "waktu_terbit": entry.get("published", datetime.now().isoformat()),
                        "sumber":       feed.feed.get("title", url),
                        "timestamp":    datetime.now().isoformat(),
                    }
                    key = hash_url(link)
                    producer_rss.send(TOPIC_RSS, key=key, value=artikel)
                    sudah_dikirim.add(link)
                    total_baru += 1
            except Exception as e:
                print(f"  [ERROR] RSS {url}: {e}")
        producer_rss.flush()
        print(f"  {total_baru} artikel baru dikirim ke {TOPIC_RSS}")
        for _ in range(INTERVAL_RSS // 5):
            if stop_flag.is_set():
                break
            time.sleep(5)
    print("[RSS Producer] Berhenti")

thread_rss = threading.Thread(target=loop_producer_rss, daemon=True, name="ProducerRSS")
thread_rss.start()
print(f" Thread producer RSS berjalan: {thread_rss.is_alive()}")

[RSS Producer] Dimulai Thread producer RSS berjalan: True


[RSS Producer] [10:37:30] Polling...


In [7]:
# buat verifikasi

from kafka import TopicPartition

def peek_topic(topic: str, n: int = 5):
    consumer = KafkaConsumer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        auto_offset_reset="earliest",
        consumer_timeout_ms=5000,
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        key_deserializer=lambda k: k.decode("utf-8") if k else None,
    )
    consumer.subscribe([topic])
    print(f"\n Isi topic '{topic}' (maks {n} event):\n")
    count = 0
    for msg in consumer:
        print(f"  offset={msg.offset} | key={msg.key} | "
              f"value={json.dumps(msg.value, ensure_ascii=False)[:120]}...")
        count += 1
        if count >= n:
            break
    if count == 0:
        print("  (kosong — tunggu producer polling pertama selesai)")
    consumer.close()

peek_topic(TOPIC_API)
peek_topic(TOPIC_RSS)


 Isi topic 'weather-api' (maks 5 event):

  0 artikel baru dikirim ke weather-rss
  (kosong — tunggu producer polling pertama selesai)

 Isi topic 'weather-rss' (maks 5 event):

  (kosong — tunggu producer polling pertama selesai)


In [8]:
# buat cek status hidup/almarhum

threads = {
    "ProducerAPI": thread_api,
    "ProducerRSS": thread_rss,
}

print("Status thread:\n")
for nama, t in threads.items():
    status = "🟢 berjalan" if t.is_alive() else "🔴 mati"
    print(f"  {nama:15s} : {status}")

Status thread:

  ProducerAPI     : 🟢 berjalan
  ProducerRSS     : 🟢 berjalan


# **3. DATA STORE**

In [9]:
import os
import subprocess

FLUSH_INTERVAL = 120   # flush ke HDFS tiap 2 menit
HDFS_API_PATH  = "/data/weather/api"
HDFS_RSS_PATH  = "/data/weather/rss"

buffer_api = []
buffer_rss = []
lock_api   = threading.Lock()
lock_rss   = threading.Lock()

def simpan_ke_hdfs(data: list, hdfs_path: str, label: str):
    if not data:
        return
    ts        = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    tmp_local = f"/tmp/weather_{label}_{ts}.json"
    tmp_container = tmp_local
    hdfs_dest = f"{hdfs_path}/{ts}.json"

    # Tulis sementara ke lokal
    with open(tmp_local, "w") as f:
        json.dump(data, f, ensure_ascii=False)

    # Copy ke container → upload ke HDFS
    subprocess.run(["docker", "cp", tmp_local, f"hadoop-namenode:{tmp_container}"], capture_output=True)
    result = subprocess.run(
        ["docker", "exec", "hadoop-namenode",
         "hdfs", "dfs", "-put", "-f", tmp_container, hdfs_dest],
        capture_output=True, text=True
    )

    if result.returncode == 0:
        print(f"  [{label}] ✅ {hdfs_dest} ({len(data)} event)")
    else:
        print(f"  [{label}] ❌ Gagal: {result.stderr.strip()}")

    os.remove(tmp_local)

def loop_consumer_api():
    consumer = KafkaConsumer(
        TOPIC_API,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id="consumer-hdfs-api",
        auto_offset_reset="earliest",
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        consumer_timeout_ms=2000,
    )
    print("[Consumer API] Dimulai")
    last_flush = time.time()
    while not stop_flag.is_set():
        for msg in consumer:
            with lock_api:
                buffer_api.append(msg.value)
            if stop_flag.is_set():
                break
        if time.time() - last_flush >= FLUSH_INTERVAL:
            with lock_api:
                data_copy = buffer_api.copy()
                buffer_api.clear()
            simpan_ke_hdfs(data_copy, HDFS_API_PATH, "API")
            last_flush = time.time()
    # flush sisa saat stop
    with lock_api:
        data_copy = buffer_api.copy()
        buffer_api.clear()
    simpan_ke_hdfs(data_copy, HDFS_API_PATH, "API")
    consumer.close()
    print("[Consumer API] Berhenti")

def loop_consumer_rss():
    consumer = KafkaConsumer(
        TOPIC_RSS,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id="consumer-hdfs-rss",
        auto_offset_reset="earliest",
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        consumer_timeout_ms=2000,
    )
    print("[Consumer RSS] Dimulai")
    last_flush = time.time()
    while not stop_flag.is_set():
        for msg in consumer:
            with lock_rss:
                buffer_rss.append(msg.value)
            if stop_flag.is_set():
                break
        if time.time() - last_flush >= FLUSH_INTERVAL:
            with lock_rss:
                data_copy = buffer_rss.copy()
                buffer_rss.clear()
            simpan_ke_hdfs(data_copy, HDFS_RSS_PATH, "RSS")
            last_flush = time.time()
    with lock_rss:
        data_copy = buffer_rss.copy()
        buffer_rss.clear()
    simpan_ke_hdfs(data_copy, HDFS_RSS_PATH, "RSS")
    consumer.close()
    print("[Consumer RSS] Berhenti")

thread_consumer_api = threading.Thread(target=loop_consumer_api, daemon=True, name="ConsumerAPI")
thread_consumer_rss = threading.Thread(target=loop_consumer_rss, daemon=True, name="ConsumerRSS")
thread_consumer_api.start()
thread_consumer_rss.start()

print(f" Consumer API berjalan : {thread_consumer_api.is_alive()}")
print(f" Consumer RSS berjalan : {thread_consumer_rss.is_alive()}")
print(f" Flush ke HDFS setiap {FLUSH_INTERVAL} detik")

 Consumer API berjalan : True
 Consumer RSS berjalan : True
 Flush ke HDFS setiap 120 detik
[Consumer RSS] Dimulai
[Consumer API] Dimulai


In [10]:
# Jalankan setelah 2 menit untuk confirm file masuk HDFS

def cek_hdfs(path):
    result = subprocess.run(
        ["docker", "exec", "hadoop-namenode", "hdfs", "dfs", "-ls", path],
        capture_output=True, text=True
    )
    print(f"📂 {path}:")
    print(result.stdout if result.stdout else "  (kosong — tunggu flush berikutnya)")

cek_hdfs("/data/weather/api")
cek_hdfs("/data/weather/rss")

📂 /data/weather/api:
  (kosong — tunggu flush berikutnya)
📂 /data/weather/rss:
  (kosong — tunggu flush berikutnya)


# **4. DATA ANALYSIS**

### Objektif


  - 1. Perbandingan suhu antar kota: rata-rata, suhu tertinggi, terendah per kota — groupBy kode_kota
  - 2. Deteksi kondisi ekstrem: filter event dengan wind_speed > 40 km/h OR humidity > 90% OR temperature > 35°C — berapa     kali dan kota mana?
  - 3. Tren suhu per jam dalam sehari: rata-rata suhu semua kota per jam — apakah ada pola diurnal?

Fokus dashboard: Tabel suhu semua kota saat ini (warna berdasarkan suhu) · Highlight kota ekstrem · Berita cuaca terbaru

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import hour, avg, max, min, col

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("WeatherAnalysis") \
    .getOrCreate()

# HDFS menyimpan data dalam format JSON, bukan Parquet. Sesuaikan URL HDFS Anda jika diperlukan.
df_api = spark.read.json("hdfs://localhost:9000/data/weather/api/")

suhu_per_kota = df_api.groupBy("nama_kota").agg(
    avg("temperature").alias("suhu_avg"),
    max("temperature").alias("suhu_tertinggi"),
    min("temperature").alias("suhu_terendah")
).orderBy("suhu_avg", ascending=False)

suhu_per_kota.show()


ModuleNotFoundError: No module named 'pyspark'

Exception in thread ProducerAPI:
Traceback (most recent call last):
  File "/home/atokdins/.local/share/uv/python/cpython-3.11.15-linux-x86_64-gnu/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/home/atokdins/.local/share/uv/python/cpython-3.11.15-linux-x86_64-gnu/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_74102/1016269063.py", line 20, in loop_producer_api
  File "/home/atokdins/Atok/big_data_college/ets/ETS-BD-5-A/.venv/lib/python3.11/site-packages/kafka/producer/kafka.py", line 844, in send
    self._wait_on_metadata(topic, timer.timeout_ms)
  File "/home/atokdins/Atok/big_data_college/ets/ETS-BD-5-A/.venv/lib/python3.11/site-packages/kafka/producer/kafka.py", line 968, in _wait_on_metadata
    raise Errors.KafkaTimeoutError(
kafka.errors.KafkaTimeoutError: KafkaTimeoutError: Failed to update metadata after 60.0 secs.


In [ ]:
from pyspark.sql.functions import count

kondisi_ekstrem = df_api.filter(
    (col("wind_speed")>40) | (col("humidity")>90) | (col("temperature")>35) 
)

rekap_ekstrem = kondisi_ekstrem.groupBy("nama_kota").agg(
    count("*").alias("jumlah_event_ekstrem")
).orderBy("jumlah_event_ekstrem",ascending=False)

rekap_ekstrem.show()
print(f"Total event atau kejadian ekstrem : {kondisi_ekstrem.count()}")

In [ ]:
from pyspark.sql.functions import hour, avg

# Memperbaiki penulisan fungsi .agg() yang sebelumnya tertulis .groupBy("temperature").alias...
tren_jam = df_api.withColumn("jam", hour(col("timestamp")))\
    .groupBy("jam")\
    .agg(avg("temperature").alias("suhu_avg"))\
    .orderBy("jam")
    
tren_jam.show(24)


# **5. DATA SERVE**